In [633]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/csv/morg08.csv")
df.head()

,hhid,intmonth,hurespli,hrhtype,minsamp,hrlonglk,hrsample,hrhhid2,serial,hhnum,...,ym_file,ym,ch02,ch35,ch613,ch1417,ch05,ihigrdc,docc00,dind02
0,2600310997690,1,2.0,1,8,2,8200,82001,1,1,...,576,561,0.0,1.0,1.0,0.0,1.0,12.0,19.0,4.0
1,2600310997690,1,2.0,1,8,2,8200,82001,1,1,...,576,561,0.0,1.0,1.0,0.0,1.0,18.0,10.0,42.0
2,7087707096191,1,2.0,1,4,2,8300,83001,1,1,...,576,573,0.0,0.0,1.0,0.0,0.0,18.0,1.0,24.0
3,7087707096191,1,2.0,1,4,2,8300,83001,1,1,...,576,573,0.0,0.0,1.0,0.0,0.0,18.0,NaN,NaN
4,41110310970391,1,1.0,1,8,2,8200,82001,1,1,...,576,561,0.0,0.0,0.0,0.0,0.0,16.0,1.0,36.0


In [634]:
df['class94'].value_counts()


class94
4.0    146008
3.0     16635
7.0     15986
5.0     14227
2.0      9829
6.0      8297
1.0      5939
8.0       221
Name: count, dtype: int64

In [635]:
df['earnwke'].isna().sum()

np.int64(142444)

In [636]:
df['uhourse'].isna().sum()

np.int64(118075)

In [637]:
#why does uhourse have a -4??
df.value_counts('uhourse')
df = df[df['uhourse'] > 0]
df = df[df['earnwke'] > 0]
#hadtouseuhourse>0toavoidhourylwagebeinginfinity

Note: Had to only use rows wehre uhourse and earnwke were > 0.

In [638]:
#Creating hourly_wage
# HOW SHOULD I TREAT NANS IN HOURLY WAGE
new_cols = pd.DataFrame({
    'hourly_wage': df['earnwke'] / df['uhourse']})
df = pd.concat([df, new_cols], axis=1)

In [639]:
#Adding log and age cutoff variables
new_cols = pd.DataFrame({'log_hourly': np.log(df["hourly_wage"]), 'age_le_30': (df["age"] <= 30), 'age_ge_55': (df["age"] >= 55)})
df = pd.concat([df, new_cols], axis=1)

In [640]:
#recoding sex variable
new_col = pd.DataFrame({'female': (df['sex'] == 2).astype(int)})
df = pd.concat([df, new_col], axis=1)


The `grade92` variable tells us highest grade completed. I created dummy variables as follows:
cateogry 1: 38 = high school no diploma
cateogry 2: 39 = hs diploma, 40 = some college but no degree
cateogry 3: 43 = bachelor's degree, 41 = associate degree occupational/vocational, 42 = associate degree academic program
category 4: 44 and above are different types of post-college degrees

In [641]:
#Recoding educational cateogories

new_cols = pd.DataFrame({
    'no_hs_diploma': (df['grade92'] <= 38).astype(int),
    'hs_some_college': df['grade92'].isin([39, 40]).astype(int),
    'college': df['grade92'].isin([41, 42, 43]).astype(int),
    'post_college': (df['grade92'] >= 44).astype(int)
})

df = pd.concat([df, new_cols], axis=1)

The codebook explains how to calculate hourly wages, which I did in the first cell: "Earnings are collected per hour for hourly workers, and per week for other workers. If you want a consistent hourly wage series during entire period, you should use earnwke/uhourse. This gives imputed hourly wage for weekly workers and actual hourly wage for hourly workers. But check earnwke for top-coding. Do not use any wage data that may be present for self-employed workers"

In [642]:
smaller_df = df[['hhid', 'state', 'county', 'class94', 'earnhre', 'age', 'female', 'grade92', 'unioncov', 'uhourse', 'weight', 'hourly_wage', 'log_hourly', 'age_le_30', 'age_ge_55', 'no_hs_diploma', 'hs_some_college', 'college', 'post_college', 'earnwke']]
smaller_df

,hhid,state,county,class94,earnhre,age,female,grade92,unioncov,uhourse,weight,hourly_wage,log_hourly,age_le_30,age_ge_55,no_hs_diploma,hs_some_college,college,post_college,earnwke
0,2600310997690,63,0,4.0,2100.0,41,0,39,2.0,40.0,2846.6161,21.000000,3.044522,False,False,0,1,0,0,840.00
1,2600310997690,63,0,5.0,2100.0,40,1,44,2.0,37.0,3118.6074,21.000000,3.044522,False,False,0,0,0,1,777.00
2,7087707096191,63,0,4.0,NaN,40,0,44,2.0,40.0,3719.7807,40.865250,3.710280,False,False,0,0,0,1,1634.61
5,41110310970391,63,73,1.0,1450.0,63,1,43,2.0,40.0,4511.5217,23.557500,3.159444,False,True,0,0,1,0,942.30
9,75680310997590,63,0,4.0,NaN,25,0,44,2.0,45.0,4067.9250,12.820444,2.551041,True,False,0,0,0,1,576.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317335,950868097156649,83,0,4.0,NaN,40,0,43,2.0,40.0,228.0247,31.250000,3.442019,False,False,0,0,1,0,1250.00
317336,950868097156649,83,0,4.0,NaN,36,0,46,2.0,40.0,231.7051,50.000000,3.912023,False,False,0,0,0,1,2000.00
317337,965567039600915,83,0,4.0,700.0,26,1,39,2.0,20.0,236.7416,7.000000,1.945910,True,False,0,1,0,0,140.00
317338,965567039600915,83,0,4.0,750.0,30,1,39,2.0,40.0,214.5578,7.500000,2.014903,True,False,0,1,0,0,300.00


In [643]:
nan_counts = smaller_df.isna().sum()
nan_counts

hhid                   0
state                  0
county                 0
class94                0
earnhre            67227
age                    0
female                 0
grade92                0
unioncov           20720
uhourse                0
weight                 0
hourly_wage            0
log_hourly             0
age_le_30              0
age_ge_55              0
no_hs_diploma          0
hs_some_college        0
college                0
post_college           0
earnwke                0
dtype: int64

Notice that `unioncov` is the main variable with a ton of NaN (and earnhre but we don't use that here)

In [644]:
#Illinois = state == 33
illinois_df = smaller_df[smaller_df['state'] == 33]
illinois_df = illinois_df.dropna()
illinois_df

,hhid,state,county,class94,earnhre,age,female,grade92,unioncov,uhourse,weight,hourly_wage,log_hourly,age_le_30,age_ge_55,no_hs_diploma,hs_some_college,college,post_college,earnwke
7652,40421370996694,33,0,2.0,1500.0,44,1,43,2.0,40.0,3149.9901,15.000000,2.708050,False,False,0,0,1,0,600.0
7658,80971090513339,33,0,4.0,1100.0,24,1,40,2.0,22.0,3775.5995,11.000000,2.397895,True,False,0,1,0,0,242.0
7659,80971090513339,33,0,4.0,1000.0,31,0,39,2.0,50.0,3997.7645,12.600000,2.533697,False,False,0,1,0,0,630.0
7669,177193559700698,33,0,4.0,1050.0,62,1,39,2.0,40.0,2692.9385,5.750000,1.749200,False,True,0,1,0,0,230.0
7689,199737302013788,33,0,4.0,1000.0,49,1,33,2.0,40.0,2967.3209,10.000000,2.302585,False,False,1,0,0,0,400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299333,771399026909396,33,0,4.0,1375.0,54,1,39,2.0,32.0,2691.8808,13.750000,2.621039,False,False,0,1,0,0,440.0
299335,829190529700969,33,0,4.0,1050.0,28,0,39,2.0,40.0,3987.0116,10.500000,2.351375,True,False,0,1,0,0,420.0
299337,867169820749105,33,115,5.0,1250.0,62,1,39,2.0,14.0,2792.0570,10.714286,2.371578,False,True,0,1,0,0,150.0
299341,902909051996827,33,0,4.0,1150.0,36,1,39,2.0,40.0,2607.0379,11.500000,2.442347,False,False,0,1,0,0,460.0


In [645]:
#IL df has 9672 rows. the below are the counts of nan
nan_counts = illinois_df.isna().sum()
nan_counts

hhid               0
state              0
county             0
class94            0
earnhre            0
age                0
female             0
grade92            0
unioncov           0
uhourse            0
weight             0
hourly_wage        0
log_hourly         0
age_le_30          0
age_ge_55          0
no_hs_diploma      0
hs_some_college    0
college            0
post_college       0
earnwke            0
dtype: int64

In [646]:
illinois_df['hourly_wage'].isna().sum()

np.int64(0)

In [647]:
illinois_df['earnhre'].isna().sum()

np.int64(0)

In [648]:
illinois_df['unioncov'].isna().sum()

np.int64(0)

In [649]:
illinois_df['uhourse'].isna().sum()

np.int64(0)

Government jobs (federal, state, local) are codes 1, 2, and 3.
Private, for profit jobs are 4.
Private, non-profit jobs are 5.

In [650]:
#Creating IL public df

illinois_public_df = illinois_df[illinois_df['class94'].isin([1, 2, 3])]

In [651]:
#Creating IL private df
illinois_private_df = illinois_df[illinois_df['class94'].isin([4, 5])]

In [652]:
#Creating other states public df
other_states_df = smaller_df[smaller_df['state'] != 33]
public_other_states_df = other_states_df[other_states_df['class94'].isin([1, 2, 3])]


In [ ]:
#new df to make Table 1
# NOTE: "other_pub" columns use public_other_states_df (class94 in [1,2,3]),
# not other_states_df (which is every non-IL respondent regardless of
# sector). Using other_states_df here was a bug -- it made the "Other
# Public" column actually report stats for all non-IL workers of any sector.
columns=['IL_pub', 'IL_priv', 'other_pub']

hourly_wage = [illinois_public_df['hourly_wage'].mean(), illinois_private_df['hourly_wage'].mean(), public_other_states_df['hourly_wage'].mean()]
log_hourly_wage = [illinois_public_df['log_hourly'].mean(), illinois_private_df['log_hourly'].mean(), public_other_states_df['log_hourly'].mean()]
age = [illinois_public_df['age'].mean(), illinois_private_df['age'].mean(), public_other_states_df['age'].mean()]
share_age_le30 = [illinois_public_df['age_le_30'].mean(), illinois_private_df['age_le_30'].mean(), public_other_states_df['age_le_30'].mean()]
share_age_ge55 = [illinois_public_df['age_ge_55'].mean(), illinois_private_df['age_ge_55'].mean(), public_other_states_df['age_ge_55'].mean()]
female = [illinois_public_df['female'].mean(), illinois_private_df['female'].mean(), public_other_states_df['female'].mean()]
no_hs_diploma = [illinois_public_df['no_hs_diploma'].mean(), illinois_private_df['no_hs_diploma'].mean(), public_other_states_df['no_hs_diploma'].mean()]
hs_some_college = [illinois_public_df['hs_some_college'].mean(), illinois_private_df['hs_some_college'].mean(), public_other_states_df['hs_some_college'].mean()]
college = [illinois_public_df['hs_some_college'].mean(), illinois_private_df['hs_some_college'].mean(), public_other_states_df['hs_some_college'].mean()]
college = [illinois_public_df['college'].mean(), illinois_private_df['college'].mean(), public_other_states_df['college'].mean()]
post_college = [illinois_public_df['post_college'].mean(), illinois_private_df['post_college'].mean(), public_other_states_df['post_college'].mean()]
weekly_hours = [illinois_public_df['uhourse'].mean(), illinois_private_df['uhourse'].mean(), public_other_states_df['uhourse'].mean()]
union_cov = [illinois_public_df['unioncov'].mean(), illinois_private_df['unioncov'].mean(), public_other_states_df['unioncov'].mean()]
n = [len(illinois_public_df), len(illinois_private_df), len(public_other_states_df)]

rows = [
    'hourly_wage',
    'log_hourly',
    'age',
    'age_le_30',
    'age_ge_55',
    'female',
    'no_hs_diploma',
    'hs_some_college',
    'college',
    'post_college',
    'uhourse',
    'unioncov', 'n'
]

all_values = [
    hourly_wage,
    log_hourly_wage,
    age,
    share_age_le30,
    share_age_ge55,
    female,
    no_hs_diploma,
    hs_some_college,
    college,
    post_college,
    weekly_hours,
    union_cov,
    n
]

table_1_df = pd.DataFrame(
    all_values,
    index=rows,
    columns=columns)

#don't forget to do p values and add to table!!

In [ ]:
#make sure variables are correct

#Setting up scipy.stats p val getting
from scipy.stats import ttest_ind 

table_1_df['pval_ILpubvpriv'] = np.nan
table_1_df['pval_ILvotherpub'] = np.nan

#need to skip the n row
for row_name, var in zip(table_1_df.index[:-1], rows[:-1]):  
    il_pub = illinois_public_df[var].dropna()
    il_priv = illinois_private_df[var].dropna()
    other = public_other_states_df[var].dropna()

    table_1_df.loc[row_name, 'pval_ILpubvpriv'] = ttest_ind(il_pub, il_priv, equal_var=False).pvalue
    table_1_df.loc[row_name, 'pval_ILvotherpub'] = ttest_ind(il_pub, other, equal_var=False).pvalue

table_1_df.loc['n', 'IL_pub'] = len(illinois_public_df)
table_1_df.loc['n', 'IL_priv'] = len(illinois_private_df)
table_1_df.loc['n', 'other_pub'] = len(public_other_states_df)
table_1_df.loc['n', ['pval_ILpubvpriv', 'pval_ILvotherpub']] = np.nan

In [655]:
#round p vals
table_1_df[['pval_ILpubvpriv', 'pval_ILvotherpub']] = \
table_1_df[['pval_ILpubvpriv', 'pval_ILvotherpub']].round(4)
table_1_df

,IL_pub,IL_priv,other_pub,pval_ILpubvpriv,pval_ILvotherpub
hourly_wage,15.026914,14.290418,20.177318,0.3419,0.0000
log_hourly,2.563610,2.526707,2.809982,0.3573,0.0000
age,42.748603,37.875396,41.118782,0.0001,0.1579
age_le_30,0.240223,0.384685,0.264721,0.0000,0.4455
age_ge_55,0.217877,0.149071,0.180877,0.0320,0.2336
female,0.653631,0.547802,0.498945,0.0049,0.0000
no_hs_diploma,0.094972,0.179429,0.095327,0.0004,0.9872
hs_some_college,0.541899,0.592207,0.483163,0.1960,0.1177
college,0.284916,0.207975,0.313540,0.0287,0.3989
post_college,0.078212,0.020390,0.107970,0.0050,0.1413


In [656]:
#How should i deal with NA?????
#ADD COMMENTS TO EVERYTHING (OR MARKDOWN)
#do the log hourly and etc up at the df level not for each

Since `unioncov` is the main variable with lots of Nan, I'm remaking table 1 below without that variable.

In [657]:
#Creating table 1 again without union cov


illinois_df = smaller_df[smaller_df['state'] == 33]
illinois_df = illinois_df.drop('unioncov', axis = 1)
illinois_df = illinois_df.dropna()

In [ ]:
illinois_public_df = illinois_df[illinois_df['class94'].isin([1, 2, 3])]
illinois_private_df = illinois_df[illinois_df['class94'].isin([4, 5])]
other_states_df = smaller_df[smaller_df['state'] != 33]
public_other_states_df = other_states_df[other_states_df['class94'].isin([1, 2, 3])]
columns=['IL_pub', 'IL_priv', 'other_pub']

# "other_pub" uses public_other_states_df (class94 in [1,2,3]), not
# other_states_df (every non-IL respondent regardless of sector).
hourly_wage = [illinois_public_df['hourly_wage'].mean(), illinois_private_df['hourly_wage'].mean(), public_other_states_df['hourly_wage'].mean()]
log_hourly_wage = [illinois_public_df['log_hourly'].mean(), illinois_private_df['log_hourly'].mean(), public_other_states_df['log_hourly'].mean()]
age = [illinois_public_df['age'].mean(), illinois_private_df['age'].mean(), public_other_states_df['age'].mean()]
share_age_le30 = [illinois_public_df['age_le_30'].mean(), illinois_private_df['age_le_30'].mean(), public_other_states_df['age_le_30'].mean()]
share_age_ge55 = [illinois_public_df['age_ge_55'].mean(), illinois_private_df['age_ge_55'].mean(), public_other_states_df['age_ge_55'].mean()]
female = [illinois_public_df['female'].mean(), illinois_private_df['female'].mean(), public_other_states_df['female'].mean()]
no_hs_diploma = [illinois_public_df['no_hs_diploma'].mean(), illinois_private_df['no_hs_diploma'].mean(), public_other_states_df['no_hs_diploma'].mean()]
hs_some_college = [illinois_public_df['hs_some_college'].mean(), illinois_private_df['hs_some_college'].mean(), public_other_states_df['hs_some_college'].mean()]
college = [illinois_public_df['hs_some_college'].mean(), illinois_private_df['hs_some_college'].mean(), public_other_states_df['hs_some_college'].mean()]
college = [illinois_public_df['college'].mean(), illinois_private_df['college'].mean(), public_other_states_df['college'].mean()]
post_college = [illinois_public_df['post_college'].mean(), illinois_private_df['post_college'].mean(), public_other_states_df['post_college'].mean()]
weekly_hours = [illinois_public_df['uhourse'].mean(), illinois_private_df['uhourse'].mean(), public_other_states_df['uhourse'].mean()]
n = [len(illinois_public_df), len(illinois_private_df), len(public_other_states_df)]

rows = [
    'hourly_wage',
    'log_hourly',
    'age',
    'age_le_30',
    'age_ge_55',
    'female',
    'no_hs_diploma',
    'hs_some_college',
    'college',
    'post_college',
    'uhourse', 'n'
]

all_values = [
    hourly_wage,
    log_hourly_wage,
    age,
    share_age_le30,
    share_age_ge55,
    female,
    no_hs_diploma,
    hs_some_college,
    college,
    post_college,
    weekly_hours,
    n
]

table_1_df = pd.DataFrame(
    all_values,
    index=rows,
    columns=columns)

In [ ]:
table_1_df['pval_ILpubvpriv'] = np.nan
table_1_df['pval_ILvotherpub'] = np.nan

#need to skip the n row
for row_name, var in zip(table_1_df.index[:-1], rows[:-1]):  
    il_pub = illinois_public_df[var].dropna()
    il_priv = illinois_private_df[var].dropna()
    other = public_other_states_df[var].dropna()

    table_1_df.loc[row_name, 'pval_ILpubvpriv'] = ttest_ind(il_pub, il_priv, equal_var=False).pvalue
    table_1_df.loc[row_name, 'pval_ILvotherpub'] = ttest_ind(il_pub, other, equal_var=False).pvalue

table_1_df.loc['n', 'IL_pub'] = len(illinois_public_df)
table_1_df.loc['n', 'IL_priv'] = len(illinois_private_df)
table_1_df.loc['n', 'other_pub'] = len(public_other_states_df)
table_1_df.loc['n', ['pval_ILpubvpriv', 'pval_ILvotherpub']] = np.nan

In [660]:
table_1_df[['pval_ILpubvpriv', 'pval_ILvotherpub']] = \
table_1_df[['pval_ILpubvpriv', 'pval_ILvotherpub']].round(4)
table_1_df

,IL_pub,IL_priv,other_pub,pval_ILpubvpriv,pval_ILvotherpub
hourly_wage,16.958338,15.215441,20.177318,0.0016,0.0000
log_hourly,2.704336,2.581956,2.809982,0.0000,0.0002
age,44.481818,38.424125,41.118782,0.0000,0.0000
age_le_30,0.187879,0.357977,0.264721,0.0000,0.0004
age_ge_55,0.239394,0.149416,0.180877,0.0003,0.0134
female,0.596970,0.501946,0.498945,0.0010,0.0003
no_hs_diploma,0.066667,0.170039,0.095327,0.0000,0.0382
hs_some_college,0.542424,0.601946,0.483163,0.0415,0.0319
college,0.321212,0.208949,0.313540,0.0000,0.7661
post_college,0.069697,0.019066,0.107970,0.0005,0.0068
